# Module T5-04: AlphaFold & Protein Design

**Tier 5 — Modern AI for Science | Module 04**

*Prerequisites: Tier 2 Module 07 (Protein Structure), Module T5-01 (LLM Fine-tuning)*

---

**By the end of this notebook you will be able to:**
1. Explain AF2 architecture at a practical level
2. Interpret pLDDT and PAE confidence metrics
3. Retrieve and inspect predicted structures from AlphaFold DB
4. Compare structures with simple RMSD and confidence-aware triage
5. Understand RFdiffusion + ProteinMPNN design loops and AF2/AF3 validation

---

[← Previous: Diffusion & Generative Models](../03_Diffusion_Generative_Models/03_Diffusion_Generative_Models.ipynb) | [Course Overview](../../README.md) | [Next: Module 05 →](../05_Genomic_Foundation_Models/)

---

## 1. Practical Architecture Overview

- **Input features**: target sequence, MSA features, template features
- **Evoformer**: iteratively updates MSA and pair representations
- **Structure module**: converts learned geometric constraints to 3D coordinates

In practice, users focus on confidence metrics and biological plausibility, not only raw coordinates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
from Bio.PDB import PDBParser, Superimposer

np.random.seed(5)

## 2. Confidence Metrics: pLDDT and PAE

- **pLDDT** (0..100): per-residue local confidence
  - >90 high, 70–90 useful, <50 often disordered
- **PAE** (angstroms): pairwise relative placement uncertainty
  - low inter-domain PAE means more reliable domain orientation

In [ ]:
# synthetic example for visualization workflow
L = 180
plddt = np.clip(np.r_[np.random.normal(88, 4, 120), np.random.normal(58, 8, 60)], 0, 100)
pae = np.random.uniform(2, 18, size=(L, L))
pae = (pae + pae.T) / 2

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(plddt, lw=1.2)
ax[0].axhline(90, ls='--', c='g', lw=1)
ax[0].axhline(70, ls='--', c='orange', lw=1)
ax[0].set_title('pLDDT profile')
ax[0].set_xlabel('Residue')
ax[0].set_ylabel('pLDDT')
im = ax[1].imshow(pae, cmap='viridis', vmin=0, vmax=30)
ax[1].set_title('PAE heatmap')
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout()

## 3. Fetching Structures from AlphaFold DB

Use UniProt IDs to fetch precomputed structures. This avoids rerunning heavy inference for many exploratory tasks.

In [ ]:
import requests
from pathlib import Path

def fetch_alphafold_pdb(uniprot_id: str, out_dir: str = 'pdb_files'):
    out = Path(out_dir)
    out.mkdir(exist_ok=True)
    url = f'https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb'
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        raise RuntimeError(f'Fetch failed: {uniprot_id} ({r.status_code})')
    path = out / f'{uniprot_id}.pdb'
    path.write_text(r.text)
    return str(path)

# Example (uncomment to run online)
# pdb_path = fetch_alphafold_pdb('P04637')  # TP53
# print(pdb_path)

## 4. Structure Comparison (RMSD)

When reference structures exist, compare predicted vs reference Cα coordinates.

In [ ]:
def ca_rmsd(pdb_a: str, pdb_b: str) -> float:
    parser = PDBParser(QUIET=True)
    s1 = parser.get_structure('a', pdb_a)
    s2 = parser.get_structure('b', pdb_b)
    ca1 = [a for a in s1.get_atoms() if a.name == 'CA']
    ca2 = [a for a in s2.get_atoms() if a.name == 'CA']
    n = min(len(ca1), len(ca2))
    if n == 0:
        raise ValueError('No CA atoms found')
    sup = Superimposer()
    sup.set_atoms(ca1[:n], ca2[:n])
    return float(sup.rms)

# Example usage:
# print(ca_rmsd('predicted.pdb', 'reference.pdb'))

## 5. Protein Design Workflow (Conceptual)

A common loop:
1. **RFdiffusion** proposes backbones
2. **ProteinMPNN** designs sequences for those backbones
3. **AF2/AF3** validates whether designed sequence recovers intended fold/interface

Use confidence filters to reduce false positives before experimental testing.

In [ ]:
def design_priority(mean_plddt: float, interface_pae: float, seq_novelty: float) -> float:
    conf = 0.7 * (mean_plddt / 100.0) + 0.3 * (1.0 - np.clip(interface_pae / 30.0, 0, 1))
    return float(0.8 * conf + 0.2 * seq_novelty)

candidates = [
    {'id': 'd1', 'plddt': 86, 'pae': 7.5, 'novelty': 0.40},
    {'id': 'd2', 'plddt': 74, 'pae': 15.0, 'novelty': 0.70},
    {'id': 'd3', 'plddt': 91, 'pae': 5.0, 'novelty': 0.20},
]
for c in candidates:
    c['priority'] = design_priority(c['plddt'], c['pae'], c['novelty'])
print(sorted(candidates, key=lambda x: x['priority'], reverse=True))

## Summary

- AF2 outputs are most useful when combined with confidence metrics.
- AlphaFold DB is a fast entry point for structure-driven hypothesis generation.
- Design pipelines should include confidence-aware ranking before experiments.